# 11 - India VIX And Macro Feature Enrichment

This notebook enriches the market regime dataset with:

1. India VIX for NIFTY
2. US macro features from FRED
3. India macro features from RBI/manual CSV exports

Outputs:

- `../data/raw/india_vix.csv`
- `../data/raw/fred_macro.csv`
- `../data/processed/features_macro_enriched.csv`

## Data Source Notes

- India VIX is available on Yahoo Finance as `^INDIAVIX`.
- FRED provides macroeconomic data through an API and Python packages such as `fredapi`.
- RBI DBIE provides Indian macro data, but direct API access is less reliable; the safest project workflow is to download selected RBI series as CSV and place them in `data/external/rbi_macro.csv`.

Suggested FRED series:

- `FEDFUNDS`: Federal Funds Rate
- `CPIAUCSL`: US Consumer Price Index
- `UNRATE`: US Unemployment Rate
- `DGS10`: 10-Year Treasury Yield
- `GDP`: US GDP

Suggested RBI/India series:

- CPI inflation
- Repo rate
- GDP growth
- 10-year government bond yield
- USD/INR exchange rate

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")

RAW_DIR = Path("../data/raw")
EXTERNAL_DIR = Path("../data/external")
PROCESSED_DIR = Path("../data/processed")

RAW_DIR.mkdir(parents=True, exist_ok=True)
EXTERNAL_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## 1. Download India VIX

Run this after installing yfinance:

```python
%pip install yfinance
```

In [ ]:
try:
    import yfinance as yf
except ModuleNotFoundError:
    raise ModuleNotFoundError("Install yfinance first: %pip install yfinance")

india_vix_raw = yf.download(
    "^INDIAVIX",
    start="2010-01-01",
    auto_adjust=True,
    progress=False,
)

print("Downloaded shape:", india_vix_raw.shape)
print("Downloaded columns:", india_vix_raw.columns)

if india_vix_raw.empty:
    raise ValueError(
        "Yahoo Finance returned no India VIX data for ^INDIAVIX. "
        "Try running yf.download('^INDIAVIX', period='max') or skip India VIX for now."
    )

# Handle both normal columns and yfinance MultiIndex columns.
if isinstance(india_vix_raw.columns, pd.MultiIndex):
    if "Close" in india_vix_raw.columns.get_level_values(0):
        close_df = india_vix_raw["Close"]
        if isinstance(close_df, pd.DataFrame):
            close_series = close_df.iloc[:, 0]
        else:
            close_series = close_df
    else:
        raise ValueError(f"Could not find Close column. Columns: {india_vix_raw.columns}")
else:
    if "Close" not in india_vix_raw.columns:
        raise ValueError(f"Could not find Close column. Columns: {india_vix_raw.columns}")
    close_series = india_vix_raw["Close"]

india_vix = close_series.rename("india_vix_close").reset_index()
india_vix = india_vix.rename(columns={india_vix.columns[0]: "Date"})
india_vix["Date"] = pd.to_datetime(india_vix["Date"])
india_vix = india_vix[["Date", "india_vix_close"]].dropna()

india_vix.to_csv(RAW_DIR / "india_vix.csv", index=False)

print("Saved:", RAW_DIR / "india_vix.csv")
print(india_vix.shape)
india_vix.head()

## 2. Download FRED Macro Data

You need a free FRED API key.

Set it in your environment before launching Jupyter:

```powershell
setx FRED_API_KEY "your_api_key_here"
```

Then restart VS Code/Jupyter.

If you do not have the key yet, skip this section and continue with India VIX first.

In [ ]:
fed_series = {
    "FEDFUNDS": "fed_funds_rate",
    "CPIAUCSL": "us_cpi",
    "UNRATE": "us_unemployment",
    "DGS10": "us_10y_yield",
    "GDP": "us_gdp",
}

fred_api_key = os.getenv("FRED_API_KEY")

if fred_api_key is None:
    print("FRED_API_KEY not found. Skipping FRED download for now.")
    fred_macro = pd.DataFrame(columns=["Date"] + list(fed_series.values()))
else:
    try:
        from fredapi import Fred
    except ModuleNotFoundError:
        raise ModuleNotFoundError("Install fredapi first: %pip install fredapi")

    fred = Fred(api_key=fred_api_key)
    series_frames = []

    for series_id, name in fed_series.items():
        s = fred.get_series(series_id)
        s = s.rename(name).reset_index()
        s.columns = ["Date", name]
        s["Date"] = pd.to_datetime(s["Date"])
        series_frames.append(s)

    fred_macro = series_frames[0]
    for frame in series_frames[1:]:
        fred_macro = fred_macro.merge(frame, on="Date", how="outer")

    fred_macro = fred_macro.sort_values("Date").reset_index(drop=True)
    fred_macro.to_csv(RAW_DIR / "fred_macro.csv", index=False)

print(fred_macro.shape)
fred_macro.tail()

## 3. Load Optional RBI Macro CSV

Create this file manually from RBI DBIE exports:

```text
../data/external/rbi_macro.csv
```

Recommended columns:

```text
Date, india_cpi, india_repo_rate, india_gdp_growth, india_10y_yield, usd_inr
```

If the file does not exist, the notebook continues without it.

In [ ]:
rbi_path = EXTERNAL_DIR / "rbi_macro.csv"

if rbi_path.exists():
    rbi_macro = pd.read_csv(rbi_path, parse_dates=["Date"])
    rbi_macro = rbi_macro.sort_values("Date").reset_index(drop=True)
else:
    print("No RBI macro file found at", rbi_path)
    rbi_macro = pd.DataFrame(columns=[
        "Date",
        "india_cpi",
        "india_repo_rate",
        "india_gdp_growth",
        "india_10y_yield",
        "usd_inr",
    ])

print(rbi_macro.shape)
rbi_macro.head()

## 4. Merge India VIX And Macro Features

We merge lower-frequency macro features using forward-fill.

Why forward-fill?

Macro variables are released monthly or quarterly, while market data is daily. The latest known macro value is carried forward until a new value is released.

In [ ]:
features = pd.read_csv(PROCESSED_DIR / "features.csv", parse_dates=["Date"])
features = features.sort_values(["ticker", "Date"]).reset_index(drop=True)

calendar = features[["Date"]].drop_duplicates().sort_values("Date")

def expand_to_daily(macro_df, calendar):
    if macro_df.empty or "Date" not in macro_df.columns:
        return calendar.copy()

    out = calendar.merge(macro_df, on="Date", how="left")
    value_cols = [c for c in out.columns if c != "Date"]
    out[value_cols] = out[value_cols].ffill()
    return out

india_vix_daily = expand_to_daily(india_vix, calendar)
fred_daily = expand_to_daily(fred_macro, calendar)
rbi_daily = expand_to_daily(rbi_macro, calendar)

macro_daily = calendar.copy()
for macro_df in [india_vix_daily, fred_daily, rbi_daily]:
    new_cols = [c for c in macro_df.columns if c != "Date" and c not in macro_daily.columns]
    macro_daily = macro_daily.merge(macro_df[["Date"] + new_cols], on="Date", how="left")

macro_daily.head()

In [ ]:
enriched = features.merge(macro_daily, on="Date", how="left")

# For NIFTY rows, prefer India VIX over US VIX when available.
enriched["asset_vix_close"] = enriched["vix_close"]
if "india_vix_close" in enriched.columns:
    nifty_mask = enriched["ticker"] == "^NSEI"
    enriched.loc[nifty_mask, "asset_vix_close"] = enriched.loc[nifty_mask, "india_vix_close"].fillna(enriched.loc[nifty_mask, "vix_close"])

# Simple macro changes useful for modeling.
for col in ["us_cpi", "us_gdp", "india_cpi", "india_repo_rate", "usd_inr"]:
    if col in enriched.columns:
        enriched[f"{col}_change_21d"] = enriched.groupby("ticker")[col].pct_change(21)

output_path = PROCESSED_DIR / "features_macro_enriched.csv"
enriched.to_csv(output_path, index=False)

print("Saved:", output_path)
print(enriched.shape)
enriched.head()

## 5. Next Modeling Step

After this notebook runs, update notebook 03 or create a new notebook to train HMMs using `asset_vix_close` instead of `vix_close`.

For NIFTY, this improves the design because India VIX is a better volatility proxy than US VIX.